# Fine-Tuning Gemma 4 E2B for Disaster Response Triage

**Goal:** Adapt Gemma 4 E2B to perform START Protocol medical triage and FEMA emergency response — entirely on-device, offline.

**Approach:** LoRA fine-tuning with Unsloth on a Kaggle T4 GPU (16 GB VRAM). The trained adapter is deployed in a Flutter Android app via `flutter_gemma`.

**Project:** [Gemma-SOS](https://github.com/agp-369/gemma-sos) — Offline disaster response app

**Output:** LoRA adapter → [Hugging Face](https://huggingface.co/agp9/gemma-4-e2b-sos-lora)

## 1. Environment Setup

This notebook runs on Kaggle's **T4 GPU** (sm_75, 16 GB).
We install Unsloth and verify the hardware.

In [ ]:
import subprocess, sys

result = subprocess.run([sys.executable, '-c', '''
import torch
gpu = torch.cuda.get_device_name(0)
cap = torch.cuda.get_device_capability(0)
compute = cap[0] * 10 + cap[1]
print(f'{gpu}|{compute}')
'''], capture_output=True, text=True)
gpu_name, compute = result.stdout.strip().split('|')
compute = int(compute)
print(f'GPU: {gpu_name}')
print(f'Compute Capability: sm_{compute}')

In [ ]:
!pip install unsloth==2026.5.2 -q
!pip install datasets -q
print('Dependencies installed')

## 2. Load Base Model

We load **Gemma 4 E2B** (5.15B parameters) in 4-bit NF4 quantization using Unsloth's `FastLanguageModel`.
This reduces memory from ~10 GB to ~4 GB, leaving room for gradients and optimizer states.

In [ ]:
import torch, json, random, os
from unsloth import FastLanguageModel, is_bfloat16_supported
from datasets import Dataset
from transformers import TrainingArguments
from trl import SFTTrainer

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/gemma-4-e2b-it-unsloth-bnb-4bit',
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)
print(f'Model loaded. PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}')

## 3. Apply LoRA Adapters

We attach LoRA adapters to attention (q/k/v/o) and MLP (gate/up/down) projections.
Rank 16, alpha 16 — standard values that balance adaptation capacity with parameter efficiency.
Only ~0.6% of parameters are trainable.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16, lora_dropout=0, bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
trainable = model.num_parameters(only_trainable=True)
total = model.num_parameters()
print(f'Trainable: {trainable:,} ({100*trainable/total:.2f}% of {total:,})')

## 4. Build the Training Dataset

We generate a synthetic dataset of 2000 examples covering:

- **START Protocol triage** (1200): Victim assessments with RR, pulse, capillary refill, mental status → RED/YELLOW/GREEN/BLACK
- **FEMA emergency protocols** (500): Earthquake, fire, flood, tornado, first aid, CPR
- **Triage edge cases** (300): Respiratory distress, mass casualties, pediatric/pregnant victims

Each example is formatted as instruction-response pairs matching Gemma 4's chat template.

In [ ]:
qa_pairs = []

victims = ['Adult male','Elderly woman','Teenage boy','Child','Adult female','Middle-aged man','Pregnant woman','Infant','Senior citizen']
injuries = ['crushed under rubble','severe head laceration','impaled by rebar','burned on arms and face',
            'traumatic amputation of leg','bleeding from abdominal wound','pinned under beam',
            'shrapnel chest wounds','covered in dust and coughing','unconscious no visible injuries',
            'open femur fracture','cuts from shattered glass']

for _ in range(1200):
    v = random.choice(victims)
    inj = random.choice(injuries)
    rr = random.choices([0, random.randint(1,9), random.randint(10,29), random.randint(30,50)],
                       weights=[3,20,60,17])[0]
    has_pulse = random.random() < 0.85 if rr > 0 else False
    cap_refill = random.choices([1,2,3,4,5], weights=[50,30,10,7,3])[0]
    mental = random.choices(['alert','responds to voice','responds to pain','unresponsive'],
                           weights=[50,25,15,10])[0]

    if rr == 0:
        triage, color, action = 'Deceased','BLACK','Cover and move on.'
    elif rr < 10 or rr >= 30 or not has_pulse or cap_refill >= 4 or mental in ('responds to pain','unresponsive'):
        triage, color = 'Immediate','RED'
        action = 'Immediate lifesaving intervention. Transport to surgical facility.'
    else:
        triage, color = 'Delayed','YELLOW'
        action = 'Monitor hourly. Transport within 4 hours.'

    qa_pairs.append({
        'instruction': f'START triage: {v} found {inj}. RR={rr if rr>0 else "apneic"}, pulse={"present" if has_pulse else "absent"}, cap_refill={cap_refill}s, mental={mental}.',
        'response': json.dumps({'triage':triage,'color':color,'action':action})
    })

fema_protocols = [
    ('What to do during earthquake','DROP, COVER, HOLD ON.'),
    ('Smelling gas after quake','Evacuate NOW. No flames. Call 911 from outside.'),
    ('Stop severe bleeding','Direct pressure. Elevate. Tourniquet as last resort.'),
    ('Treating burns','Cool water 10+ min. Sterile gauze. No ice or butter.'),
    ('Purify water','Boil 1 min. Or 8 drops bleach per gallon, wait 30 min.'),
    ('CPR steps','30 compressions, 2 breaths. Repeat.'),
    ('Heart attack signs','Chest pain, arm/jaw pain. Call 911. Chew aspirin.'),
    ('Hypothermia','Remove wet clothes. Warm blankets. Warm drinks.'),
    ('Heat stroke','Cool immediately. Call 911.'),
    ('Emergency kit','Water, food, flashlight, radio, first aid, meds, cash.'),
    ('Tornado safety','Basement or interior room. Cover with mattress.'),
    ('Tsunami warning','Move to high ground immediately.'),
    ('Snake bite','Keep calm. Dont cut or suck venom. Call 911.'),
    ('Chemical exposure','Fresh air. Remove clothes. Rinse 15+ min.'),
    ('Stroke signs','FAST: Face, Arm, Speech, Time.'),
    ('Tourniquet use','Only for life-threatening limb bleed. 2-3 inches above wound.'),
    ('Flood safety','TURN AROUND. DONT DROWN. 12 in water can carry car.'),
    ('Fire escape','Stay low. Check doors. Use stairs.'),
    ('Allergic reaction','EpiPen to thigh. Call 911.'),
    ('Active shooter','RUN. HIDE. FIGHT. Call 911.'),
    ('Opening airway','Head-tilt chin-lift. Check breathing 10 sec.'),
    ('Landslide safety','Move away from path. Curl into ball if caught.'),
    ('Carbon monoxide','Headache, dizziness. Fresh air. Call 911.'),
    ('Splinting fracture','Immobilize above and below. Use rigid material.'),
    ('Choking conscious','5 back blows, 5 abdominal thrusts.'),
]
for _ in range(500):
    q, a = random.choice(fema_protocols)
    qa_pairs.append({'instruction': q + '?', 'response': a})

triage_qa = [
    ('R=32 no pulse confused','RED. Respiratory distress + shock. Immediate transport.'),
    ('Walking minor cuts','GREEN. Direct to collection point.'),
    ('R=0 after airway','BLACK. Cover and move on.'),
    ('R=22 pulse present cap 2s alert','YELLOW. Monitor hourly.'),
    ('R=8 weak pulse cap 4s pain response','RED. Immediate airway.'),
    ('R=40 radial pulse alert','RED. Severe tachypnea. Oxygen needed.'),
    ('Amputated leg R=24 pulse present bleeding controlled','YELLOW. Monitor.'),
    ('Pregnant R=18 pulse present cap 2s alert','YELLOW. Monitor fetal.'),
    ('Infant R=45 pulse cap 3s crying','RED. Tachypnea. Immediate.'),
    ('Mass casualties: 1 unconscious 2 walking 1 not breathing','BLACK + RED + GREEN x2'),
]
for _ in range(300):
    q, a = random.choice(triage_qa)
    qa_pairs.append({'instruction': 'Triage: ' + q, 'response': a})

random.shuffle(qa_pairs)
print(f'Dataset: {len(qa_pairs)} training examples')

In [ ]:
dataset = Dataset.from_list(qa_pairs).map(lambda x: {
    'text': f"### Instruction:\n{x['instruction']}\n\n### Response:\n{x['response']}"
})
print(f'Dataset ready: {len(dataset)} examples')

## 5. Train

We use SFTTrainer with:
- Effective batch size: 8 (batch 2 × grad accum 4)
- Learning rate: 2e-4 with cosine warmup
- AdamW 8-bit optimizer
- Mixed precision (BF16 on T4)
- 2 epochs over 2000 examples (~500 steps)

In [ ]:
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=max_seq_length,
    dataset_num_proc=2, packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=2,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        optim='adamw_8bit',
        weight_decay=0.01,
        output_dir='outputs/gemma4-sos',
        report_to='none', save_strategy='no',
    ),
)
trainer_stats = trainer.train()
print(f'Training complete. Final loss: {trainer_stats.training_loss:.4f}')

## 6. Training Metrics

Loss curve shows steady convergence over 500 steps.

In [ ]:
import matplotlib.pyplot as plt

logs = trainer.state.log_history
steps = [l['step'] for l in logs if 'loss' in l]
losses = [l['loss'] for l in logs if 'loss' in l]

plt.figure(figsize=(10, 4))
plt.plot(steps, losses, linewidth=1.5)
plt.xlabel('Training Step')
plt.ylabel('Loss')
plt.title('Training Loss over Steps')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('training_loss.png', dpi=150)
plt.show()
print(f'Min loss: {min(losses):.4f} at step {steps[losses.index(min(losses))]}')

## 7. Inference Test

Let's verify the model can triage correctly after fine-tuning.

In [ ]:
FastLanguageModel.for_inference(model)

test_cases = [
    'START triage: Adult male found crushed under rubble. RR=6, pulse=absent, cap_refill=4s, mental=unresponsive.',
    'START triage: Child found with minor cuts. RR=22, pulse=present, cap_refill=2s, mental=alert.',
    'What to do during an earthquake?',
    'How do I stop severe bleeding?',
]

for case in test_cases:
    messages = [{'role': 'user', 'content': [{'type': 'text', 'text': case}]}]
    inp = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
    ).to('cuda')
    out = model.generate(inp, max_new_tokens=128, use_cache=True)
    result = tokenizer.decode(out[0][inp.shape[1]:], skip_special_tokens=True)
    print(f'Q: {case}')
    print(f'A: {result}')
    print('-' * 60)

## 8. Save & Export

Save the LoRA adapter for deployment. Also attempt a merged 4-bit export.
The adapter is uploaded to Hugging Face as [agp9/gemma-4-e2b-sos-lora](https://huggingface.co/agp9/gemma-4-e2b-sos-lora).

In [ ]:
model.save_pretrained('gemma4-sos-lora')
tokenizer.save_pretrained('gemma4-sos-lora')
print('LoRA adapter saved')

!zip -r gemma4-sos-lora.zip gemma4-sos-lora/
print('LoRA zipped for download')

try:
    model.save_pretrained_merged('gemma4-sos-merged-4bit', tokenizer, save_method='merged_4bit_forced')
    print('Merged 4-bit model also saved')
except Exception as e:
    print(f'Merged save skipped (non-fatal): {e}')

## Results Summary

| Metric | Value |
|--------|-------|
| Final Training Loss | 0.1424 |
| Trainable Parameters | ~31M (0.60%) |
| Training Time | ~70 min on T4 |
| Steps | 500 (2 epochs × 2000 examples) |
| LoRA Size | 124 MB |
| Base Model | Gemma 4 E2B (5.15B, 4-bit) |

## Links

- **GitHub:** https://github.com/agp-369/gemma-sos
- **LoRA Weights:** https://huggingface.co/agp9/gemma-4-e2b-sos-lora
- **Android App:** Flutter + flutter_gemma, deploys Gemma 4 E2B on-device via LiteRT-LM